# Consultas SQL desde PostgreSQL



In [ ]:
import pandas as pd
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
from metrobus.load import get_engine

engine = get_engine()
conn = engine.connect()

print("Conexi?n correcta a PostgreSQL")

## 1 Cuales son los vehículos mas caros de mantener?


In [3]:
query_1 = """
SELECT
    v.vehiculo_id,
    v.modelo,
    ROUND(SUM(m.coste_eur)::numeric, 2) AS coste_total_mantenimiento
FROM fact_mantenimiento m
JOIN dim_vehiculo v
    ON m.vehiculo_id = v.vehiculo_id
GROUP BY v.vehiculo_id,  v.modelo
ORDER BY coste_total_mantenimiento DESC
LIMIT 15;
"""

df_1 = pd.read_sql(query_1, conn)
df_1

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\1252310641.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_1 = pd.read_sql(query_1, conn)


,vehiculo_id,modelo,coste_total_mantenimiento
0,14,Mercedes-Benz eCitaro,31760.82
1,43,MAN Lions City G,26735.24
2,6,Solaris Urbino 18,25891.04
3,22,BYD eBus-12,24216.84
4,15,Scania Citywide,24184.68
5,8,BYD eBus-12,23684.06
6,33,Iveco Urbanway,23121.76
7,42,MAN Lions City,22832.86
8,17,Mercedes-Benz Citaro,22165.29
9,31,Mercedes-Benz Citaro,20145.96


## 2 Cuales son los vehiculos que mas mantenimientos necesitan?


In [4]:
query_2 = """
SELECT
    v.vehiculo_id,
    v.modelo,
    COUNT(m.mantenimiento_id) AS numero_mantenimientos
FROM fact_mantenimiento m
JOIN dim_vehiculo v
    ON m.vehiculo_id = v.vehiculo_id
GROUP BY v.vehiculo_id,  v.modelo
ORDER BY numero_mantenimientos DESC
LIMIT 15;
"""

df_2 = pd.read_sql(query_2, conn)
df_2

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\562244228.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_2 = pd.read_sql(query_2, conn)


,vehiculo_id,modelo,numero_mantenimientos
0,19,Iveco Urbanway,30
1,18,Iveco Urbanway,30
2,26,Volvo 7900 Hybrid,30
3,22,BYD eBus-12,29
4,6,Solaris Urbino 18,29
5,42,MAN Lions City,29
6,43,MAN Lions City G,28
7,31,Mercedes-Benz Citaro,28
8,14,Mercedes-Benz eCitaro,26
9,13,MAN Lions City G,25


## 3  Cual es la antigüedad media del personal?


In [5]:
query_3 = """
SELECT
    ROUND(AVG(antiguedad_anos)::numeric, 2) AS antiguedad_media_anos
FROM dim_conductor;
"""

df_3 = pd.read_sql(query_3, conn)
df_3

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\3492655853.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_3 = pd.read_sql(query_3, conn)


,antiguedad_media_anos
0,10.5


## 4  Cuantos viajes realiza un conductor al año?


In [6]:
query_4 = """
SELECT
    EXTRACT(YEAR FROM v.fecha)::int AS anio,
    c.conductor_id,
    c.nombre,
    COUNT(DISTINCT v.viaje_id) AS total_viajes
FROM fact_viajes v
JOIN dim_conductor c
    ON v.conductor_id = c.conductor_id
GROUP BY EXTRACT(YEAR FROM v.fecha), c.conductor_id, c.nombre
ORDER BY anio, total_viajes DESC;
"""

df_4 = pd.read_sql(query_4, conn)
df_4

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\3514877239.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_4 = pd.read_sql(query_4, conn)


,anio,conductor_id,nombre,total_viajes
0,2022,10,Carmen Ruiz,637
1,2022,24,Patricia Serrano,633
2,2022,27,Victor Blanco,625
3,2022,23,Raul Ramos,623
4,2022,3,Juan Martinez,622
...,...,...,...,...
79,2024,8,Isabel Perez,577
80,2024,21,Alejandro Castro,576
81,2024,19,Jorge Alonso,568
82,2024,16,Marta Hernandez,564


## 5 Que tasa de incidencias tiene cada conductor por cada 100 viajes?


In [ ]:
query_5 = """
WITH viajes_por_conductor AS (
    SELECT
        EXTRACT(YEAR FROM fecha)::int AS anio,
        conductor_id,
        COUNT(DISTINCT viaje_id) AS total_viajes
    FROM fact_viajes
    GROUP BY EXTRACT(YEAR FROM fecha), conductor_id
),
incidencias_por_conductor AS (
    SELECT
        EXTRACT(YEAR FROM fecha)::int AS anio,
        conductor_id,
        COUNT(DISTINCT incidencia_id) AS total_incidencias
    FROM fact_incidencias
    GROUP BY EXTRACT(YEAR FROM fecha), conductor_id
)
SELECT
    v.anio,
    c.conductor_id,
    c.nombre,
    v.total_viajes,
    COALESCE(i.total_incidencias, 0) AS total_incidencias,
    ROUND(
        100.0 * COALESCE(i.total_incidencias, 0)
        / NULLIF(v.total_viajes, 0),
        2
    ) AS incidencias_por_100_viajes
FROM viajes_por_conductor v
JOIN dim_conductor c
    ON v.conductor_id = c.conductor_id
LEFT JOIN incidencias_por_conductor i
    ON i.anio = v.anio
    AND i.conductor_id = v.conductor_id
ORDER BY v.anio, incidencias_por_100_viajes DESC;
"""

df_5 = pd.read_sql(query_5, conn)
df_5

## 6 Cual es el retraso medio de salida por linea?


In [ ]:
query_6 = """
SELECT
    l.linea_id,
    l.codigo,
    ROUND(AVG(GREATEST(v.retraso_salida_min, 0))::numeric, 2)
        AS retraso_medio_min_por_viaje,
    ROUND(
        100.0 * AVG((v.retraso_salida_min <= 5)::int)::numeric,
        2
    ) AS puntualidad_pct
FROM fact_viajes v
JOIN dim_linea l
    ON v.linea_id = l.linea_id
GROUP BY l.linea_id, l.codigo
ORDER BY retraso_medio_min_por_viaje DESC;
"""

df_6 = pd.read_sql(query_6, conn)
df_6

## 7 Cuales son los tipos de incidentes mas frecuentes?


In [10]:
query_7 = """
SELECT
    tipo_incidencia,
    COUNT(*) AS total_incidencias
FROM fact_incidencias
GROUP BY tipo_incidencia
ORDER BY total_incidencias DESC;
"""

df_7 = pd.read_sql(query_7, conn)
df_7

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\237144204.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_7 = pd.read_sql(query_7, conn)


,tipo_incidencia,total_incidencias
0,Condiciones meteo,382
1,Averia mecanica,347
2,Obras calzada,340
3,Huelga parcial,339
4,Pinchazo,335
5,Accidente leve,332
6,Fallo electrico,332
7,Vandalismo,330
8,Accidente grave,328
9,Fallo climatizacion,312


## 8 Que paradas tienen mas retraso medio de salida?


In [ ]:
query_8 = """
SELECT
    p.parada_id,
    p.nombre_parada,
    COUNT(DISTINCT v.viaje_id) AS total_viajes,
    ROUND(AVG(GREATEST(v.retraso_salida_min, 0))::numeric, 2)
        AS retraso_medio_min_por_viaje,
    ROUND(
        100.0 * AVG((v.retraso_salida_min <= 5)::int)::numeric,
        2
    ) AS puntualidad_pct
FROM fact_viajes v
JOIN dim_parada p
    ON v.parada_origen_id = p.parada_id
GROUP BY p.parada_id, p.nombre_parada
ORDER BY retraso_medio_min_por_viaje DESC
LIMIT 15;
"""

df_8 = pd.read_sql(query_8, conn)
df_8

## 9 Cuantos dias al año pasa cada vehiculo en mantenimiento? (incluye vehiculos sin intervenciones; 2025 es parcial)


In [ ]:
query_9 = """
WITH periodos AS (
    SELECT
        EXTRACT(YEAR FROM fecha_entrada)::int AS anio,
        MIN(fecha_entrada) AS inicio_periodo,
        MAX(fecha_entrada) AS fin_periodo
    FROM fact_mantenimiento
    GROUP BY EXTRACT(YEAR FROM fecha_entrada)
),
dias_por_vehiculo AS (
    SELECT
        p.anio,
        p.inicio_periodo,
        p.fin_periodo,
        v.vehiculo_id,
        COALESCE(SUM(m.dias_fuera_servicio), 0) AS dias_mantenimiento
    FROM periodos p
    CROSS JOIN dim_vehiculo v
    LEFT JOIN fact_mantenimiento m
        ON m.vehiculo_id = v.vehiculo_id
        AND EXTRACT(YEAR FROM m.fecha_entrada)::int = p.anio
    GROUP BY p.anio, p.inicio_periodo, p.fin_periodo, v.vehiculo_id
)
SELECT
    d.anio,
    MIN(d.inicio_periodo) AS primera_fecha_mantenimiento,
    MAX(d.fin_periodo) AS ultima_fecha_mantenimiento,
    COUNT(*) AS vehiculos_incluidos,
    ROUND(AVG(dias_mantenimiento)::numeric, 2)
        AS promedio_dias_mantenimiento_por_vehiculo
FROM dias_por_vehiculo d
GROUP BY d.anio
ORDER BY d.anio;
"""

df_9 = pd.read_sql(query_9, conn)
df_9

## 10 Cuantos pasajeros lleva cada linea en cada viaje en promedio?


In [15]:
query_10 = """
SELECT
    l.linea_id,
    l.codigo,
    COUNT(DISTINCT v.viaje_id) AS total_viajes,
    ROUND(AVG(v.pasajeros_subidos)::numeric, 2)
        AS pasajeros_promedio_por_viaje
FROM fact_viajes v
JOIN dim_linea l
    ON v.linea_id = l.linea_id
GROUP BY l.linea_id, l.codigo
ORDER BY pasajeros_promedio_por_viaje DESC;
"""

df_10 = pd.read_sql(query_10, conn)
df_10

C:\Users\Ncasl\AppData\Local\Temp\ipykernel_28640\1641260647.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_10 = pd.read_sql(query_10, conn)


,linea_id,codigo,total_viajes,pasajeros_promedio_por_viaje
0,4,L4,4840,64.60
1,9,L9,5024,64.51
2,1,L1,4954,63.98
3,8,L8,5003,63.89
4,2,L2,5073,63.80
5,3,L3,4949,63.67
6,7,L7,5129,63.47
7,5,L5,5128,63.19
8,6,L6,4990,62.81
9,10,N1,4910,38.58


In [ ]:
conn.close()
engine.dispose()